# Etapas 4+5 — Split Temporal e Modelagem (v2 — corrigido)

**Correções aplicadas vs. v1:**
- SARIMA → SARIMAX com `exog=[onpromotion, oil_price]` (comparação justa)
- XGBoost: early stopping usa validação interna (últimos 90 dias de `train_full`), **não** o conjunto de teste
- Documentado: Prophet, XGBoost e SARIMAX operam em modo de *forecast condicional* — recebem valores reais dos regressores no período de teste

**Split:** Últimos 90 dias como teste (sem random split).

In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

PROCESSED_DIR = '../data/processed/'
DATA_DIR = '../data/'

daily = pd.read_csv(PROCESSED_DIR + 'beverages_daily.csv', parse_dates=['date'])
daily = daily.sort_values('date').reset_index(drop=True)

features_df = pd.read_csv(PROCESSED_DIR + 'beverages_features.csv', parse_dates=['date'])
features_df = features_df.sort_values('date').reset_index(drop=True)

print(f'Série diária: {len(daily)} obs, {daily.date.min().date()} a {daily.date.max().date()}')
print(f'Features: {len(features_df)} obs, {features_df.date.min().date()} a {features_df.date.max().date()}')

Série diária: 1684 obs, 2013-01-01 a 2017-08-15
Features: 1319 obs, 2014-01-02 a 2017-08-15


## Etapa 4 — Split Temporal

In [2]:
TEST_DAYS = 90

train_full = daily.iloc[:-TEST_DAYS].copy()
test_full  = daily.iloc[-TEST_DAYS:].copy()

train_feat = features_df.iloc[:-TEST_DAYS].copy()
test_feat  = features_df.iloc[-TEST_DAYS:].copy()

train_end  = train_full['date'].max()
test_start = test_full['date'].min()
test_end   = test_full['date'].max()

print(f'Treino: {len(train_full)} obs → até {train_end.date()}')
print(f'Teste:  {len(test_full)} obs → {test_start.date()} a {test_end.date()}')

y_test     = test_full['sales'].values
dates_test = test_full['date'].values

Treino: 1594 obs → até 2017-05-17
Teste:  90 obs → 2017-05-18 a 2017-08-15


## Etapa 5 — Modelagem

### 5.1 Naive

In [3]:
last_train_value = train_full['sales'].iloc[-1]
naive_pred = np.full(TEST_DAYS, last_train_value)
print(f'Naive — previsão constante: {last_train_value:.0f}')

Naive — previsão constante: 165675


### 5.2 Sazonal Naive (s=7)

In [4]:
full_series = daily['sales'].values
train_size  = len(train_full)

seasonal_naive_pred = np.array([
    full_series[train_size + i - 7] for i in range(TEST_DAYS)
])

print(f'Sazonal Naive — primeiras 7 previsões: {seasonal_naive_pred[:7].round(0)}')

Sazonal Naive — primeiras 7 previsões: [135028. 154026. 244193. 206986. 179114. 169098. 165675.]


### 5.3 SARIMAX (1,1,1)(1,1,1,7) com exog=[onpromotion, oil_price]

**Nota:** As previsões usam valores reais de `onpromotion` e `oil_price` no período de teste (*forecast condicional*). Isso é equivalente ao que Prophet e XGBoost fazem — garante comparação justa entre os três modelos.

In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog_cols  = ['onpromotion', 'oil_price']
exog_train = train_full[exog_cols].values
exog_test  = test_full[exog_cols].values

print('Treinando SARIMAX(1,1,1)(1,1,1,7) com exog=[onpromotion, oil_price]...')
t0 = time.time()

sarima_model = SARIMAX(
    train_full['sales'],
    exog=exog_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
t1 = time.time()

print(f'Concluído em {t1-t0:.1f}s  |  AIC: {sarima_fit.aic:.2f}  |  BIC: {sarima_fit.bic:.2f}')

Treinando SARIMAX(1,1,1)(1,1,1,7) com exog=[onpromotion, oil_price]...


Concluído em 1.8s  |  AIC: 36239.25  |  BIC: 36276.79


In [6]:
sarima_pred = np.maximum(sarima_fit.forecast(steps=TEST_DAYS, exog=exog_test).values, 0)
print(f'SARIMAX — primeiras 7 previsões: {sarima_pred[:7].round(0)}')
print(f'Valores reais:                   {y_test[:7].round(0)}')

SARIMAX — primeiras 7 previsões: [145049. 162791. 237079. 244516. 186457. 159491. 171911.]
Valores reais:                   [133381. 161718. 227900. 280849. 169143. 151437. 153706.]


### 5.4 Prophet (regressores obrigatórios + feriados + terremoto)

**Nota (forecast condicional):** O Prophet recebe os valores reais de `onpromotion` e `oil_price` no `make_future_dataframe` do período de teste.

In [7]:
from prophet import Prophet

prophet_train = train_full[['date','sales','onpromotion','oil_price']].copy()
prophet_train.columns = ['ds','y','onpromotion','oil_price']

holidays_raw = pd.read_csv(DATA_DIR + 'holidays_events.csv', parse_dates=['date'])
nat_holidays = holidays_raw[
    (holidays_raw['locale'] == 'National') &
    (holidays_raw['transferred'] == False)
][['date','description']].drop_duplicates()
nat_holidays.columns = ['ds','holiday']

eq_df = pd.DataFrame({
    'ds': pd.date_range('2016-04-16', periods=14, freq='D'),
    'holiday': 'Terremoto_Equador_2016'
})
holidays_prophet = pd.concat([nat_holidays, eq_df], ignore_index=True)
print(f'Feriados + eventos para Prophet: {len(holidays_prophet)} entradas')

Feriados + eventos para Prophet: 180 entradas


In [8]:
print('Treinando Prophet...')
t0 = time.time()

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    holidays=holidays_prophet
)
prophet_model.add_regressor('onpromotion')
prophet_model.add_regressor('oil_price')
prophet_model.fit(prophet_train)

t1 = time.time()
print(f'Concluído em {t1-t0:.1f}s')

22:44:42 - cmdstanpy - INFO - Chain [1] start processing


Treinando Prophet...


22:44:42 - cmdstanpy - INFO - Chain [1] done processing


Concluído em 0.4s


In [9]:
future = test_full[['date','onpromotion','oil_price']].copy()
future.columns = ['ds','onpromotion','oil_price']

prophet_pred = np.maximum(prophet_model.predict(future)['yhat'].values, 0)

print(f'Prophet — primeiras 7 previsões: {prophet_pred[:7].round(0)}')
print(f'Valores reais:                   {y_test[:7].round(0)}')

Prophet — primeiras 7 previsões: [153639. 169347. 216926. 233409. 178489. 168167. 174022.]
Valores reais:                   [133381. 161718. 227900. 280849. 169143. 151437. 153706.]


### 5.5 XGBoost (early stopping sem look-ahead bias)

**Protocolo:**
1. Dentro de `train_feat`, separar os últimos 90 dias como `val` para early stopping
2. Treinar com early stopping usando `val` (não o teste)
3. Retreinar no `train_feat` completo com o `best_iteration` encontrado
4. Prever em `test_feat` — completamente isolado

In [10]:
import xgboost as xgb

FEATURE_COLS = [
    'year','month','day','day_of_week','day_of_year','week_of_year','quarter',
    'lag_7','lag_14','lag_30','lag_365',
    'rolling_7_mean','rolling_30_mean','rolling_7_std','rolling_30_std',
    'onpromotion','oil_price','is_national_holiday'
]

# Validação interna: últimos 90 dias de train_feat
VAL_DAYS = 90
train_sub_feat = train_feat.iloc[:-VAL_DAYS].copy()
val_sub_feat   = train_feat.iloc[-VAL_DAYS:].copy()

X_train_sub = train_sub_feat[FEATURE_COLS].values
y_train_sub = train_sub_feat['sales'].values
X_val       = val_sub_feat[FEATURE_COLS].values
y_val       = val_sub_feat['sales'].values
X_test_xgb  = test_feat[FEATURE_COLS].values
y_test_xgb  = test_feat['sales'].values

print(f'train_sub: {X_train_sub.shape}  |  val: {X_val.shape}  |  test: {X_test_xgb.shape}')
print(f'Período val:  {val_sub_feat["date"].min().date()} a {val_sub_feat["date"].max().date()}')
print(f'Período test: {test_feat["date"].min().date()} a {test_feat["date"].max().date()}')

train_sub: (1139, 18)  |  val: (90, 18)  |  test: (90, 18)
Período val:  2017-02-17 a 2017-05-17
Período test: 2017-05-18 a 2017-08-15


In [11]:
print('Passo 1: encontrar best_iteration via early stopping em val (não no teste)...')
t0 = time.time()

xgb_cv = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_cv.fit(
    X_train_sub, y_train_sub,
    eval_set=[(X_val, y_val)],
    verbose=False
)
best_n = xgb_cv.best_iteration
print(f'Best iteration (via val): {best_n}')

print(f'\nPasso 2: retreinar em train_full com n_estimators={best_n}...')
X_train_full = train_feat[FEATURE_COLS].values
y_train_full = train_feat['sales'].values

xgb_model = xgb.XGBRegressor(
    n_estimators=best_n,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train_full, y_train_full)
t1 = time.time()

print(f'Concluído em {t1-t0:.1f}s  |  n_estimators finais: {best_n}')

Passo 1: encontrar best_iteration via early stopping em val (não no teste)...


Best iteration (via val): 583

Passo 2: retreinar em train_full com n_estimators=583...


Concluído em 1.5s  |  n_estimators finais: 583


In [12]:
xgb_pred = np.maximum(xgb_model.predict(X_test_xgb), 0)

print(f'XGBoost — primeiras 7 previsões: {xgb_pred[:7].round(0)}')
print(f'Valores reais:                   {y_test_xgb[:7].round(0)}')

XGBoost — primeiras 7 previsões: [141421. 152072. 224501. 234681. 168583. 157871. 156579.]
Valores reais:                   [133381. 161718. 227900. 280849. 169143. 151437. 153706.]


## 5.6 Salvar previsões e artefatos

In [13]:
predicoes = pd.DataFrame({
    'date':          dates_test,
    'y_real':        y_test,
    'naive':         naive_pred,
    'sazonal_naive': seasonal_naive_pred,
    'sarima':        sarima_pred,
    'prophet':       prophet_pred,
    'xgboost':       xgb_pred
})

predicoes.to_csv(PROCESSED_DIR + 'predicoes_teste.csv', index=False)
print(f'predicoes_teste.csv salvo  |  Shape: {predicoes.shape}')
predicoes.head(10)

predicoes_teste.csv salvo  |  Shape: (90, 7)


,date,y_real,naive,sazonal_naive,sarima,prophet,xgboost
0,2017-05-18,133381.0,165675.0,135028.0,145049.199554,153639.353205,141421.296875
1,2017-05-19,161718.0,165675.0,154026.0,162791.218200,169347.163497,152072.078125
2,2017-05-20,227900.0,165675.0,244193.0,237079.103408,216925.814039,224501.218750
3,2017-05-21,280849.0,165675.0,206986.0,244516.418176,233408.505758,234680.937500
4,2017-05-22,169143.0,165675.0,179114.0,186457.257759,178488.826184,168582.515625
5,2017-05-23,151437.0,165675.0,169098.0,159491.344964,168167.286660,157871.203125
6,2017-05-24,153706.0,165675.0,165675.0,171911.358528,174022.174373,156578.812500
7,2017-05-25,136779.0,165675.0,133381.0,135585.745195,156574.435945,135235.875000
8,2017-05-26,212870.0,165675.0,161718.0,162521.300085,198903.037044,167193.437500
9,2017-05-27,212127.0,165675.0,227900.0,236152.080392,220144.889558,233251.859375


In [14]:
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df.to_csv(PROCESSED_DIR + 'xgb_feature_importance.csv', index=False)
print('Feature importance XGBoost (top 10):')
print(importance_df.head(10).to_string(index=False))

Feature importance XGBoost (top 10):
            feature  importance
              lag_7    0.294679
        day_of_week    0.127446
     rolling_7_mean    0.121088
        onpromotion    0.098675
             lag_14    0.093202
      rolling_7_std    0.054132
    rolling_30_mean    0.043295
is_national_holiday    0.038877
                day    0.022524
        day_of_year    0.019997


In [15]:
prophet_full_input = pd.concat([
    prophet_train[['ds','onpromotion','oil_price']],
    future[['ds','onpromotion','oil_price']]
], ignore_index=True)

prophet_components = prophet_model.predict(prophet_full_input)
prophet_components.to_csv(PROCESSED_DIR + 'prophet_components.csv', index=False)
print(f'prophet_components.csv salvo  |  Shape: {prophet_components.shape}')

prophet_components.csv salvo  |  Shape: (1684, 250)
